<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>From Scratch MLP - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [18]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [19]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [20]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [21]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [22]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [23]:
internet_required = False

In [24]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

In [25]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [26]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [27]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
43405,34,blue-collar,married,secondary,no,800,no,no,cellular,5,apr,144,1,-1,0,unknown,yes
20677,48,services,married,secondary,no,260,no,no,cellular,12,aug,228,5,-1,0,unknown,no
45164,35,services,married,tertiary,no,4655,no,no,cellular,9,nov,111,2,-1,0,unknown,no
34500,51,management,married,tertiary,no,630,yes,no,cellular,5,may,138,1,-1,0,unknown,no
20288,39,management,married,secondary,no,215,no,no,cellular,11,aug,1141,4,-1,0,unknown,yes
29603,60,retired,married,tertiary,no,3461,no,no,cellular,3,feb,128,2,-1,0,unknown,no
29196,48,admin.,single,secondary,no,303,yes,yes,cellular,2,feb,142,1,189,4,failure,no
17783,47,management,married,tertiary,no,2889,no,no,telephone,29,jul,259,12,-1,0,unknown,no
8578,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
29966,40,self-employed,married,tertiary,no,1498,no,no,cellular,4,feb,62,1,-1,0,unknown,no


In [28]:
df.dropna(inplace=True)
df.shape

(45211, 17)

# Conversion and Encoding Ready for the MLP

xscaled​=max(x)−min(x)x−min(x)​

In [29]:
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Binary columns that should not be one-hot encoded
binary_columns = ['default', 'housing', 'loan', 'y']

# Columns to exclude from the system
excluded_features = ['day', 'campaign', 'pdays', 'previous', 'y']

numerical_to_normalize = [col for col in numerical_cols if col not in binary_columns]
categorical_to_normalize = [col for col in categorical_cols if col not in binary_columns]

### Define Normalisation and Label Encoding Functions

$scaled_x = \frac{max_x - min_x}{x-min_x}$

In [31]:
def max_min_normalize (df, column_name):
    max_value = df[column_name].max()
    min_value = df[column_name].min()
    df[column_name] = (df[column_name] - min_value) / (max_value - min_value)
    return df

def label_encode(df, column_name):
    # Get unique values in the column
    unique_values = df[column_name].unique()
    
    # Create a mapping of unique values to integers
    mapping = {value: i for i, value in enumerate(unique_values)}
    
    # Apply the mapping to the column
    df[column_name] = df[column_name].map(mapping).astype(float)
    return df

In [ ]:
# Clean up the binary columns to be 0 and 1
for col in binary_columns:
    df[col] = df[col].map({'yes': 1.0, 'no': 0.0}).astype(float)

# Normalize the categorical columns using label encoding and then max-min normalization
for col in categorical_to_normalize:
    df = label_encode(df, col)
    df = max_min_normalize(df, col)

# Normalize the numerical columns using max-min normalization
for col in numerical_to_normalize:
    df = max_min_normalize(df, col)


In [33]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
19008,0.389610,0.090909,0.0,0.333333,0.0,0.087330,0.0,0.0,0.5,0.133333,0.272727,0.023790,0.000000,0.000000,0.000000,0.0,0.0
33912,0.090909,0.000000,0.5,0.000000,0.0,0.081365,0.0,0.0,0.5,0.966667,0.909091,0.104107,0.000000,0.000000,0.000000,0.0,1.0
27144,0.220779,0.272727,0.0,0.333333,0.0,0.072803,1.0,1.0,0.5,0.666667,0.454545,0.022977,0.000000,0.000000,0.000000,0.0,0.0
8362,0.168831,0.272727,0.0,0.333333,0.0,0.068754,0.0,0.0,0.0,0.066667,0.090909,0.036600,0.000000,0.000000,0.000000,0.0,0.0
34064,0.376623,0.000000,0.5,0.000000,0.0,0.073584,0.0,0.0,0.5,0.966667,0.909091,0.078691,0.032258,0.100917,0.003636,1.0,1.0
11935,0.428571,0.000000,0.0,0.000000,0.0,0.070397,1.0,0.0,0.0,0.633333,0.090909,0.088451,0.048387,0.000000,0.000000,0.0,0.0
36748,0.129870,0.545455,0.0,0.333333,0.0,0.072522,1.0,0.0,0.5,0.366667,0.000000,0.075031,0.000000,0.000000,0.000000,0.0,0.0
8547,0.324675,0.545455,0.0,0.333333,0.0,0.076698,1.0,1.0,0.0,0.066667,0.090909,0.007117,0.500000,0.000000,0.000000,0.0,0.0
16537,0.428571,0.000000,0.0,0.000000,0.0,0.277595,1.0,0.0,1.0,0.733333,0.181818,0.014437,0.032258,0.000000,0.000000,0.0,0.0
5511,0.168831,0.090909,0.5,0.000000,0.0,0.076671,1.0,0.0,0.0,0.733333,0.000000,0.016673,0.096774,0.000000,0.000000,0.0,0.0


In [34]:
sys.exit('Stopping notebook run here.')    

SystemExit: Stopping notebook run here.

---

### Write out the cleaned dataset for further use



In [ ]:
# Create filenames, one timestamped, one 'latest'
filepath = "."
filename = f'{filepath}/bank_full_cleaned_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_cleaned_latest.csv'

# Write out CSV files
df.to_csv(filename, index=False)
df.to_csv(latest_filename, index=False)